# 1.Load the CSV File

In [2]:
import pandas as pd

In [3]:
import numpy as np

In [4]:
df=pd.read_csv("C:/Users/abhin/Downloads/Data Set/amazon_products_sales_data_uncleaned.csv")

# 2.Inspect the Data

df.head()

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   title                     42675 non-null  object
 1   rating                    41651 non-null  object
 2   number_of_reviews         41651 non-null  object
 3   bought_in_last_month      39458 non-null  object
 4   current/discounted_price  30926 non-null  object
 5   price_on_variant          42675 non-null  object
 6   listed_price              42675 non-null  object
 7   is_best_seller            42675 non-null  object
 8   is_sponsored              42675 non-null  object
 9   is_couponed               42675 non-null  object
 10  buy_box_availability      28022 non-null  object
 11  delivery_details          30955 non-null  object
 12  sustainability_badges     3408 non-null   object
 13  image_url                 42675 non-null  object
 14  product_url           

In [6]:
df.shape

(42675, 16)

In [7]:
df.describe(include="all")

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
count,42675,41651,41651,39458,30926,42675,42675,42675,42675,42675,28022,30955,3408,42675,40606,42675
unique,8808,31,4413,59,2576,3466,911,12,2,42,1,298,16,8038,40606,331
top,"Duracell Coppertop 9V Battery, 6 Count (Pack o...",4.6 out of 5 stars,25,100+ bought in past month,29.99,basic variant price: nan,No Discount,No Badge,Organic,No Coupon,Add to cart,"Delivery Mon, Sep 1",Small Business,https://m.media-amazon.com/images/I/51Xr76m2WL...,/Lorex-8-Channel-Security-Outdoor-Cameras/dp/B...,25-08-2025 11:07
freq,744,6151,626,8801,825,21031,30364,40814,35664,40727,28022,6189,1341,1461,1,180


In [8]:
df.columns

Index(['title', 'rating', 'number_of_reviews', 'bought_in_last_month',
       'current/discounted_price', 'price_on_variant', 'listed_price',
       'is_best_seller', 'is_sponsored', 'is_couponed', 'buy_box_availability',
       'delivery_details', 'sustainability_badges', 'image_url', 'product_url',
       'collected_at'],
      dtype='object')

# 3.Rename column headers


In [9]:
df=df.rename(columns={
    'title':'product_title',
    'rating':'product_rating',
    'number_of_reviews':'total_reviews',
    'bought_in_last_month':'purchased_last_month',
    'current/discounted_price':'discounted_price',
    'listed_price':'original_price',
    'is_couponed':'has_coupon',
    'delivery_details':'delivery_date',
    'sustainability_badges':'sustainability_tags',
    'image_url':'product_image_url',
    'product_url':'product_page_url',
    'collected_at':'date_collected_at'
})
    

In [10]:
df.columns

Index(['product_title', 'product_rating', 'total_reviews',
       'purchased_last_month', 'discounted_price', 'price_on_variant',
       'original_price', 'is_best_seller', 'is_sponsored', 'has_coupon',
       'buy_box_availability', 'delivery_date', 'sustainability_tags',
       'product_image_url', 'product_page_url', 'date_collected_at'],
      dtype='object')

In [11]:
df.isnull().sum()


product_title               0
product_rating           1024
total_reviews            1024
purchased_last_month     3217
discounted_price        11749
price_on_variant            0
original_price              0
is_best_seller              0
is_sponsored                0
has_coupon                  0
buy_box_availability    14653
delivery_date           11720
sustainability_tags     39267
product_image_url           0
product_page_url         2069
date_collected_at           0
dtype: int64

# 4. Remove unnecessary whitespace


In [12]:
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

 # Convert blank strings to NaN

In [13]:
 df[c] = df[c].replace(r"^\s*$", np.nan, regex=True)

# 5. Clean Rating.

In [14]:
df["rating_value"] = pd.to_numeric(
    df["product_rating"]
    .astype("string")
    .str.extract(r"([0-5](?:\.\d+)?)", expand=False),
    errors="coerce"
)

# 6. Clean number of reviews.

In [15]:
df["number_of_reviews_value"] = pd.to_numeric(
    df["total_reviews"]
    .astype("string")
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# 7. Clean monthly purchases

In [16]:
df["bought_last_month_min"] = pd.to_numeric(
    df["purchased_last_month"]
    .astype("string")
    .str.extract(r"([\d,]+(?:\.\d+)?)", expand=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# 8. Function for cleaning prices

In [17]:
def clean_price(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(r"[$,]", "", regex=True),
        errors="coerce"
    )


df["discounted_price"] = clean_price(
    df["discounted_price"]
)

df["original_price"] = clean_price(
    df["original_price"]
)

# 9. Clean variant price

In [18]:
df["variant_price_value"] = pd.to_numeric(
    df["price_on_variant"]
    .astype("string")
    .str.extract(r"\$([\d,]+(?:\.\d+)?)", expand=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# 10. Convert Sponsored/Organic to Boolean

In [19]:
df["is_sponsored_flag"] = df["is_sponsored"].map({
    "Sponsored": True,
    "Organic": False
})


# 11. Best seller flag

In [20]:
df["is_best_seller_flag"] = (
    df["is_best_seller"] == "Best Seller"
)



# 12. Coupon flag

In [21]:
df["has_coupon_flag"] = (
    df["has_coupon"].notna()
    & df["has_coupon"].ne("No Coupon")
)


# 13. Handle missing categorical values.

In [22]:
for column in [
    "buy_box_availability",
    "delivery_date",
    "sustainability_tags"
]:
    df[column] = df[column].fillna("Not specified")

# 14. Convert collected_at to datetime.

In [23]:
df["date_collected_at"] = pd.to_datetime(
    df["date_collected_at"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

# 15. Create date and time columns.

In [24]:
df["collected_date"] = df["date_collected_at"].dt.strftime("%Y-%m-%d")
df["collected_time"] = df["date_collected_at"].dt.strftime("%H:%M:%S")

# 16. Calculate discount.

In [25]:
df["discount_amount"] = (
    df["original_price"]
    - df["discounted_price"]
)


# 17. Calculate discount percentage

In [26]:
df["discount_pct"] = np.where(
    df["original_price"].notna()
    & (df["original_price"] > 0)
    & df["discounted_price"].notna(),

    df["discount_amount"]
    / df["original_price"]
    * 100,

    np.nan
)



# 18. Remove impossible negative discounts.

In [27]:
df.loc[
    df["discount_amount"] < 0,
    ["discount_amount", "discount_pct"]
] = np.nan

# 19. Round percentage.

In [28]:
df["discount_pct"] = df["discount_pct"].round(2)

# 20. Remove duplicate rows.

In [29]:
df = df.drop_duplicates().reset_index(drop=True)

# 21. Save cleaned dataset.

In [32]:
df.to_csv(
    "amazon_products_sales_data_cleaned_by_abhinav.csv",
    index=False,encoding='utf-8'
)